# 🚀 LLM Inference Optimization: From Theory to Practice
### A Hands-On Lab for Deep Learning Researchers and Engineers

### Author : Manoj Kumar
---

## 📋 Overview

Large Language Models (LLMs) like LLaMA-3 are extraordinarily powerful — but deploying them at scale is expensive and slow. This lab walks through **six industry-standard inference optimization techniques** that every production ML engineer must know:

| # | Technique | Key Benefit |
|---|-----------|-------------|
| 1 | **KV Cache** | Eliminates redundant attention recomputation |
| 2 | **Batching (Static, Dynamic, Continuous)** | Maximizes GPU utilization |
| 3 | **Quantization (FP32→FP16→INT8→INT4)** | Reduces memory footprint 2–8× |
| 4 | **Flash Attention** | O(n) memory attention; 2–4× faster |
| 5 | **Speculative Decoding** | 2–3× decode throughput boost |
| 6 | **Continuous Batching (vLLM)** | Production-grade serving throughput |

---

## 🎯 Learning Objectives

By the end of this lab you will be able to:
- Identify **where** inference bottlenecks occur (prefill vs decode, memory bandwidth vs compute)
- Implement and benchmark each optimization technique on **WikiText-2**
- Reason about quality-efficiency tradeoffs using perplexity, latency, and throughput
- Choose the **right combination** of techniques for a given deployment scenario

---

## 🖥️ Hardware Requirements

| Tier | Hardware | Supported Sections |
|------|----------|--------------------|
| **Minimum** | CPU only | Sec 1–4 (small batch) |
| **Recommended** | Single GPU ≥ 16 GB VRAM (e.g. A10G, T4×2) | All sections |
| **Ideal** | A100 40GB+ | Speculative decoding at scale |

> **Note:** All cells include CPU fallbacks. GPU cells are guarded with `torch.cuda.is_available()` checks.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 ── Install dependencies
# Run once; restart kernel afterwards if installing for the first time
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "torch>=2.2.0",
    "accelerate>=0.29.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.8.0",
    "pandas>=2.2.0",
    "seaborn>=0.13.0",
    "sentencepiece",
    "protobuf",
    "tqdm",
)

# Flash-Attention 2 — only install if CUDA is available
try:
    import torch
    if torch.cuda.is_available():
        pip_install("flash-attn", "--no-build-isolation")
        print("✅ flash-attn installed")
    else:
        print("⚠️  CUDA not available – flash-attn skipped (CPU fallback will be used in Section 5)")
except Exception as e:
    print(f"⚠️  flash-attn install failed: {e}")

# vLLM — optional, Section 3c only
try:
    pip_install("vllm>=0.4.0")
    print("✅ vllm installed")
except Exception as e:
    print(f"⚠️  vllm not installed (Section 3c will show a conceptual demo): {e}")

print("\n✅ Core dependencies ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 ── Global imports & configuration
# ─────────────────────────────────────────────────────────────────────────────
import os, time, gc, math, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# HuggingFace
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
)
from datasets import load_dataset

# ── Seaborn theme ────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.2)
COLORS = sns.color_palette("muted", 8)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"🖥️  Device : {DEVICE}")
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU    : {gpu_name}  ({total_mem:.1f} GB VRAM)")
print(f"   dtype  : {DTYPE}")

# ── Model choice ─────────────────────────────────────────────────────────────
# Using Llama-3.2-1B (public, ungated). Switch to a larger variant if you have the VRAM.
# For the speculative decoding section a 3B target model is also used.
MODEL_ID       = "meta-llama/Llama-3.2-1B"
MODEL_ID_LARGE = "meta-llama/Llama-3.2-3B"  # target model for speculative decoding

# ── Shared results store ─────────────────────────────────────────────────────
# All benchmark cells write here; Section 7 builds the final dashboard from it.
RESULTS: Dict[str, dict] = {}

print("\n✅ Imports OK")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 ── Load WikiText-2 & build a fixed evaluation slice
# ─────────────────────────────────────────────────────────────────────────────
print("📥 Loading WikiText-2 (wikitext-2-raw-v1) ...")
wikitext = load_dataset("wikitext", "wikitext-2-raw-v1")

# Build a list of non-trivial sentences from the test split
raw_texts = [t.strip() for t in wikitext["test"]["text"]
             if len(t.strip()) > 80]          # skip headings / blank lines

print(f"   Total usable test sentences : {len(raw_texts)}")

# ── Fixed 100-prompt evaluation slice (deterministic) ────────────────────────
np.random.seed(42)
EVAL_PROMPTS: List[str] = list(
    np.random.choice(raw_texts, size=min(100, len(raw_texts)), replace=False)
)
# Truncate each prompt to its first 120 characters so latency stays predictable
EVAL_PROMPTS = [p[:120] for p in EVAL_PROMPTS]

print(f"   Evaluation slice size        : {len(EVAL_PROMPTS)} prompts")
print(f"\n📝 Sample prompt:\n   » {EVAL_PROMPTS[0][:90]}...")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 ── Global benchmark utilities
# ─────────────────────────────────────────────────────────────────────────────

def sync():
    """GPU sync before timing."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def gpu_mem_gb() -> float:
    """Current GPU memory allocated in GB (0 if CPU-only)."""
    if DEVICE == "cuda":
        return torch.cuda.memory_allocated() / 1e9
    return 0.0

def peak_gpu_mem_gb() -> float:
    """Peak GPU memory allocated in GB."""
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    return 0.0

def reset_peak_mem():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

def benchmark_generation(
    model,
    tokenizer,
    prompts: List[str],
    max_new_tokens: int = 128,
    use_cache: bool = True,
    batch_size: int = 1,
    label: str = "run",
    **gen_kwargs,
) -> dict:
    """
    Runs autoregressive generation on a list of prompts and returns:
      - latency_ms      : mean per-sample latency (ms)
      - throughput_tps  : tokens generated per second
      - peak_mem_gb     : peak GPU memory (GB)
      - ttft_ms         : time-to-first-token across samples (mean)
    """
    model.eval()
    reset_peak_mem()
    latencies, ttfts, total_new_tokens = [], [], 0

    for i in tqdm(range(0, len(prompts), batch_size),
                  desc=f"  [{label}]", leave=False):
        batch = prompts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt",
                        padding=True, truncation=True,
                        max_length=256).to(DEVICE)

        with torch.no_grad():
            # ── Time-to-first-token ──────────────────────────────────────────
            sync()
            t0 = time.perf_counter()
            _ = model(**enc)          # single forward pass ≈ prefill
            sync()
            ttfts.append((time.perf_counter() - t0) * 1000)

            # ── Full generation latency ──────────────────────────────────────
            sync()
            t1 = time.perf_counter()
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                use_cache=use_cache,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                **gen_kwargs,
            )
            sync()
            elapsed = time.perf_counter() - t1

        n_new = (out.shape[-1] - enc["input_ids"].shape[-1]) * len(batch)
        latencies.append(elapsed * 1000 / len(batch))   # ms / sample
        total_new_tokens += n_new

    total_time_s = sum(latencies) * len(prompts) / 1000  # rough total
    throughput = total_new_tokens / max(total_time_s, 1e-6)

    return {
        "label"          : label,
        "latency_ms"     : float(np.mean(latencies)),
        "throughput_tps" : float(throughput),
        "peak_mem_gb"    : peak_gpu_mem_gb(),
        "ttft_ms"        : float(np.mean(ttfts)),
    }


def compute_perplexity(model, tokenizer, texts: List[str],
                       stride: int = 512) -> float:
    """Compute perplexity on a list of texts using sliding-window NLL."""
    model.eval()
    nlls, n_tokens = [], 0
    for text in tqdm(texts[:20], desc="  [ppl]", leave=False):  # limit to 20 for speed
        enc = tokenizer(text, return_tensors="pt").to(DEVICE)
        seq_len = enc.input_ids.size(1)
        if seq_len < 2:
            continue
        with torch.no_grad():
            out = model(**enc, labels=enc.input_ids)
        nlls.append(out.loss.item() * (seq_len - 1))
        n_tokens += seq_len - 1
    return math.exp(sum(nlls) / n_tokens) if n_tokens > 0 else float("nan")


def bar_chart(data: dict, title: str, ylabel: str, color_offset=0):
    fig, ax = plt.subplots(figsize=(8, 4))
    keys, vals = list(data.keys()), list(data.values())
    bars = ax.bar(keys, vals,
                  color=[COLORS[(i + color_offset) % len(COLORS)]
                         for i in range(len(keys))],
                  edgecolor="white", linewidth=0.8)
    ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=10)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()


print("✅ Benchmark utilities ready")

---
## Section 1 — Why Inference Optimization Matters

Before optimizing anything, we must understand *where* the cost is. This section establishes baselines on **WikiText-2** using a vanilla LLaMA-3.2-1B model in FP16.

### Key concepts

**Prefill phase** — the model processes the entire input prompt in parallel (like an encoder forward pass). This is **compute-bound**.

**Decode phase** — the model generates one token at a time, each step loading all model weights from GPU HBM. This is **memory-bandwidth bound**.

> The roofline model tells us: for modern A100 GPUs (2 TB/s memory bandwidth, 312 TFLOP/s), Llama-3.2-1B at FP16 (2.4 GB of weights) can only sustain ~16 tokens/s per request — far below the theoretical compute ceiling. **We are bandwidth-bound.**

### What we'll measure
- Baseline latency (ms/sample) and throughput (tokens/s)
- Time-to-first-token (TTFT) vs prompt length
- GPU memory growth with sequence length

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1a ── Load baseline model (FP16)
# ─────────────────────────────────────────────────────────────────────────────
print(f"📥 Loading tokenizer & model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # LLaMA-3 has no pad token by default

model_baseline = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE == "cpu":
    model_baseline = model_baseline.to(DEVICE)
model_baseline.eval()

param_count = sum(p.numel() for p in model_baseline.parameters()) / 1e6
model_size_gb = sum(p.numel() * p.element_size()
                    for p in model_baseline.parameters()) / 1e9

print(f"\n   Parameters : {param_count:.0f} M")
print(f"   Model size : {model_size_gb:.2f} GB ({DTYPE})")
print("✅ Baseline model loaded")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1b ── Baseline generation benchmark on WikiText-2
# ─────────────────────────────────────────────────────────────────────────────
baseline_results = benchmark_generation(
    model_baseline, tokenizer,
    prompts=EVAL_PROMPTS[:20],        # first 20 prompts for speed
    max_new_tokens=64,
    use_cache=True,
    batch_size=1,
    label="Baseline FP16",
)

print("\n📊 Baseline Benchmark (WikiText-2, 20 prompts, 64 new tokens):")
print(f"   Latency (mean)     : {baseline_results['latency_ms']:.1f} ms / sample")
print(f"   Throughput         : {baseline_results['throughput_tps']:.1f} tokens/sec")
print(f"   Time-to-first-token: {baseline_results['ttft_ms']:.1f} ms")
print(f"   Peak GPU memory    : {baseline_results['peak_mem_gb']:.2f} GB")

# Store for final dashboard
RESULTS["Baseline (FP16)"] = {
    "latency_ms"    : baseline_results["latency_ms"],
    "throughput_tps": baseline_results["throughput_tps"],
    "peak_mem_gb"   : baseline_results["peak_mem_gb"],
    "perplexity"    : None,   # computed next
    "notes"         : "No optimization",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1c ── Baseline perplexity on WikiText-2
# ─────────────────────────────────────────────────────────────────────────────
print("📐 Computing baseline perplexity on WikiText-2 test set ...")
baseline_ppl = compute_perplexity(model_baseline, tokenizer, EVAL_PROMPTS[:20])
print(f"\n   WikiText-2 Perplexity : {baseline_ppl:.2f}")
RESULTS["Baseline (FP16)"]["perplexity"] = baseline_ppl

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1d ── TTFT vs prompt length & memory growth visualization
# ─────────────────────────────────────────────────────────────────────────────
prompt_lengths = [64, 128, 256, 512]
ttft_results   = {"lengths": prompt_lengths, "ttft_ms": [], "mem_gb": []}

base_text  = " ".join(EVAL_PROMPTS[:10])   # long source text to slice from
tokens_src = tokenizer.encode(base_text)

print("⏱️  Measuring TTFT vs prompt length ...")
for plen in prompt_lengths:
    # Build a prompt of exactly `plen` tokens
    ids = torch.tensor([tokens_src[:plen]], device=DEVICE)
    attn = torch.ones_like(ids)

    reset_peak_mem()
    with torch.no_grad():
        sync()
        t0 = time.perf_counter()
        _ = model_baseline(input_ids=ids, attention_mask=attn)
        sync()
        ttft_ms = (time.perf_counter() - t0) * 1000
    ttft_results["ttft_ms"].append(ttft_ms)
    ttft_results["mem_gb"].append(peak_gpu_mem_gb())
    print(f"   prompt_len={plen:4d}  TTFT={ttft_ms:.1f} ms  peak_mem={ttft_results['mem_gb'][-1]:.2f} GB")

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(prompt_lengths, ttft_results["ttft_ms"],
             marker="o", color=COLORS[0], linewidth=2.5)
axes[0].set_title("Time-to-First-Token vs Prompt Length", fontweight="bold")
axes[0].set_xlabel("Prompt Length (tokens)")
axes[0].set_ylabel("TTFT (ms)")

axes[1].plot(prompt_lengths, ttft_results["mem_gb"],
             marker="s", color=COLORS[3], linewidth=2.5)
axes[1].set_title("Peak GPU Memory vs Prompt Length", fontweight="bold")
axes[1].set_xlabel("Prompt Length (tokens)")
axes[1].set_ylabel("GPU Memory (GB)")

plt.suptitle("Section 1 — Baseline Profiling on WikiText-2",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\n🔍 Observation: TTFT scales roughly O(n²) in standard attention,")
print("   and memory scales linearly with sequence length.")

### 📝 Assignment 1 — Baseline Profiling

**Task:** Profile TTFT and throughput across a wider range of prompt lengths.

1. Measure **TTFT** and **throughput (tokens/sec)** at prompt lengths `[64, 128, 256, 512, 1024]`.
2. Plot TTFT vs prompt length and fit a trend line. Does it scale **O(n)** or **O(n²)**?
3. Compute the **arithmetic intensity** (FLOPs / byte) for the prefill phase at each length using the formula:
   $$\text{Arithmetic Intensity} = \frac{4 \cdot n \cdot d_{model}^2}{2 \cdot P \cdot B}$$
   where $n$ = sequence length, $d_{model}$ = hidden dim, $P$ = parameter count, $B$ = bytes per parameter.
4. At which sequence length does the model cross from memory-bandwidth bound to compute-bound?

**Starter code is in the cell below. Fill in the `# YOUR CODE HERE` sections.**

In [ ]:
# ── Assignment 1 Starter Code ─────────────────────────────────────────────────
assignment_lengths = [64, 128, 256, 512, 1024]
a1_ttft   = []
a1_tps    = []

for plen in assignment_lengths:
    ids  = torch.tensor([tokens_src[:plen]], device=DEVICE)
    attn = torch.ones_like(ids)

    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # 1. Measure TTFT (single forward pass, no generation)
    # 2. Measure throughput: generate 64 new tokens and compute tokens/sec
    # Hint: use sync() before/after, time.perf_counter(), and model.generate()

    ttft_ms   = None   # replace with your measurement
    tps       = None   # replace with your measurement
    # ────────────────────────────────────────────────────────────────────────

    a1_ttft.append(ttft_ms)
    a1_tps.append(tps)

# ── Plot ─────────────────────────────────────────────────────────────────────
# YOUR CODE HERE: plot a1_ttft vs assignment_lengths and a1_tps vs assignment_lengths
# Add a trend-line to the TTFT plot using np.polyfit

print("✅ Assignment 1 complete — discuss your observations in the cell below")

# ── 💡 Solution hint (collapsed) ─────────────────────────────────────────────
# TTFT timing:
#   sync(); t0 = time.perf_counter()
#   with torch.no_grad(): model(input_ids=ids, attention_mask=attn)
#   sync(); ttft_ms = (time.perf_counter()-t0)*1000
#
# Throughput:
#   sync(); t1 = time.perf_counter()
#   out = model.generate(ids, max_new_tokens=64, use_cache=True, do_sample=False)
#   sync(); elapsed = time.perf_counter() - t1
#   tps = 64 / elapsed

---
## Section 2 — KV Cache

### What is the KV Cache?

In autoregressive decoding, at each step $t$ the model computes **keys** and **values** for all previous tokens just to attend over them again. Without caching, this is $O(T^2)$ total work for a sequence of length $T$.

The **KV Cache** stores these intermediate tensors so each decode step only computes keys/values for the **single new token**, reducing decode complexity to $O(T)$.

$$\text{KV cache size} = 2 \times n_{\text{layers}} \times n_{\text{heads}} \times d_{\text{head}} \times T \times \text{bytes\_per\_element}$$

For LLaMA-3.2-1B:
- 16 layers, 8 KV heads (GQA), $d_{head}$ = 64, FP16 = 2 bytes
- At T=512: $2 \times 16 \times 8 \times 64 \times 512 \times 2 = 16\,\text{MB}$
- At T=4096: $2 \times 16 \times 8 \times 64 \times 4096 \times 2 = 128\,\text{MB}$

### Grouped Query Attention (GQA) — LLaMA-3 specific
LLaMA-3 uses **GQA** (Grouped Query Attention): multiple query heads share a single key-value head group, shrinking the KV cache by $n_q / n_{kv}$ ×.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2a ── KV Cache: with vs. without
# ─────────────────────────────────────────────────────────────────────────────
print("⚡ Benchmarking KV Cache ON vs OFF on WikiText-2 ...")
print("   (use_cache=True is the HuggingFace default)\n")

gen_kwargs = dict(
    prompts       = EVAL_PROMPTS[:10],
    max_new_tokens= 64,
    batch_size    = 1,
)

# With KV cache (default)
res_cache_on = benchmark_generation(
    model_baseline, tokenizer,
    use_cache=True, label="KV Cache ON", **gen_kwargs)

# Without KV cache (recomputes all KV at every decode step)
res_cache_off = benchmark_generation(
    model_baseline, tokenizer,
    use_cache=False, label="KV Cache OFF", **gen_kwargs)

# ── Print comparison ──────────────────────────────────────────────────────────
print("\n📊 KV Cache Comparison:")
print(f"{'':30s}  {'KV ON':>10s}  {'KV OFF':>10s}  {'Speedup':>8s}")
print("-"*62)

for metric, fmt in [("latency_ms", "{:.1f} ms"), ("throughput_tps", "{:.1f} t/s"),
                    ("ttft_ms", "{:.1f} ms"), ("peak_mem_gb", "{:.2f} GB")]:
    on_val  = res_cache_on[metric]
    off_val = res_cache_off[metric]
    # For latency, lower is better; for throughput, higher is better
    if metric == "throughput_tps":
        speedup = on_val / max(off_val, 1e-6)
    else:
        speedup = off_val / max(on_val, 1e-6)
    label_name = metric.replace("_", " ").title()
    print(f"  {label_name:28s}  {fmt.format(on_val):>10s}  "
          f"{fmt.format(off_val):>10s}  {speedup:>7.2f}×")

RESULTS["KV Cache"] = {
    "latency_ms"    : res_cache_on["latency_ms"],
    "throughput_tps": res_cache_on["throughput_tps"],
    "peak_mem_gb"   : res_cache_on["peak_mem_gb"],
    "perplexity"    : RESULTS["Baseline (FP16)"]["perplexity"],  # no quality change
    "notes"         : "use_cache=True (HF default)",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2b ── KV Cache memory footprint formula + visualization
# ─────────────────────────────────────────────────────────────────────────────
cfg = model_baseline.config

# GQA-aware KV head count
n_kv_heads = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
n_layers   = cfg.num_hidden_layers
d_head     = cfg.hidden_size // cfg.num_attention_heads
bytes_fp16 = 2

seq_lengths  = list(range(64, 4097, 64))
kv_cache_mb  = [
    (2 * n_layers * n_kv_heads * d_head * T * bytes_fp16) / 1e6
    for T in seq_lengths
]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(seq_lengths, kv_cache_mb, color=COLORS[1], linewidth=2.5)
ax.fill_between(seq_lengths, kv_cache_mb, alpha=0.15, color=COLORS[1])
ax.axvline(512,  color="grey", linestyle="--", alpha=0.7, label="T=512")
ax.axvline(2048, color="orangered", linestyle="--", alpha=0.7, label="T=2048")
ax.set_title(f"KV Cache Memory Footprint\n"
             f"(LLaMA-3.2-1B, {n_layers} layers, {n_kv_heads} KV heads, FP16)",
             fontweight="bold")
ax.set_xlabel("Sequence Length (tokens)")
ax.set_ylabel("KV Cache Size (MB)")
ax.legend()

# Annotate a few points
for t in [512, 1024, 2048, 4096]:
    mb_val = (2 * n_layers * n_kv_heads * d_head * t * bytes_fp16) / 1e6
    ax.annotate(f"{mb_val:.0f} MB",
                xy=(t, mb_val), xytext=(t + 80, mb_val + 2),
                fontsize=9, arrowprops=dict(arrowstyle="->", color="grey"))

plt.tight_layout()
plt.show()

print(f"\n📌 Config: n_layers={n_layers}, n_kv_heads={n_kv_heads}, "
      f"d_head={d_head}, dtype=FP16 (2B)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2c ── Visualize KV cache speedup
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Latency comparison
labels   = ["KV Cache ON", "KV Cache OFF"]
latencies = [res_cache_on["latency_ms"], res_cache_off["latency_ms"]]
throughputs = [res_cache_on["throughput_tps"], res_cache_off["throughput_tps"]]

bars0 = axes[0].bar(labels, latencies, color=[COLORS[2], COLORS[6]], edgecolor="white")
axes[0].bar_label(bars0, fmt="%.1f ms", padding=4)
axes[0].set_title("Latency per Sample (lower is better)", fontweight="bold")
axes[0].set_ylabel("ms / sample")

bars1 = axes[1].bar(labels, throughputs, color=[COLORS[2], COLORS[6]], edgecolor="white")
axes[1].bar_label(bars1, fmt="%.1f t/s", padding=4)
axes[1].set_title("Throughput (higher is better)", fontweight="bold")
axes[1].set_ylabel("Tokens / sec")

speedup = res_cache_on["throughput_tps"] / max(res_cache_off["throughput_tps"], 1e-6)
plt.suptitle(f"KV Cache Speedup: {speedup:.2f}× on WikiText-2",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 📝 Assignment 2 — Sliding-Window KV Cache Eviction

In practice, GPU memory is finite. When the KV cache exceeds a budget, the simplest strategy is a **sliding window**: keep only the last $W$ token positions in the cache.

**Task:**

1. Implement a **sliding-window token generation loop** that manually trims `past_key_values` to the last `window_size` tokens after each decode step.

2. Generate continuations for 10 WikiText-2 prompts with window sizes `W ∈ [32, 64, 128, 256, full]`.

3. For each window size, compute:
   - **WikiText-2 perplexity** on the generated continuation
   - **Peak GPU memory** consumed

4. Plot the tradeoff: **Perplexity vs. window size** and **Memory vs. window size**.

5. At which window size does perplexity degrade noticeably? What does this tell us about **attention locality** in language models?

> 💡 Hint: `past_key_values` is a tuple of `(key, value)` per layer. Each tensor has shape `[batch, heads, seq_len, d_head]`. You can slice `[..., -W:, :]` to retain only the last W positions.

In [ ]:
# ── Assignment 2 Starter Code ─────────────────────────────────────────────────
def sliding_window_generate(model, tokenizer, prompt: str,
                             max_new_tokens: int = 64,
                             window_size: Optional[int] = None) -> str:
    """
    Autoregressive generation with optional KV cache window eviction.
    window_size=None means full cache (no eviction).
    """
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids   = enc["input_ids"]
    past_kv     = None
    generated   = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            if past_kv is None:
                out = model(input_ids=input_ids, use_cache=True)
            else:
                # Only feed the last token; use cached past
                out = model(input_ids=input_ids[:, -1:],
                            past_key_values=past_kv, use_cache=True)

            past_kv = out.past_key_values
            # ── YOUR CODE HERE ────────────────────────────────────────────────
            # If window_size is not None, trim past_kv to the last window_size
            # positions along dim=-2 (sequence dimension).
            # past_kv is a tuple of length n_layers,
            # each element is (k, v) with shape [B, H, T, d_head]
            # ──────────────────────────────────────────────────────────────────

            next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=-1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated, skip_special_tokens=True)


# ── Test with a single prompt ─────────────────────────────────────────────────
test_prompt = EVAL_PROMPTS[0]
gen_full   = sliding_window_generate(model_baseline, tokenizer,
                                     test_prompt, max_new_tokens=32,
                                     window_size=None)
gen_window = sliding_window_generate(model_baseline, tokenizer,
                                     test_prompt, max_new_tokens=32,
                                     window_size=16)

print("Full cache output  :", gen_full[:120])
print("Window-16 output   :", gen_window[:120])
print("\n✅ Assignment 2 scaffold ready — implement window trimming above")

---
## Section 3 — Batching: Static, Dynamic, and Continuous

### Why batch requests?

Running requests **one at a time** leaves most of the GPU idle. The GPU is designed for massively parallel workloads — combining multiple requests into a batch lets us amortize weight loads across many samples.

| Strategy | Description | Weakness |
|----------|-------------|----------|
| **Static batching** | Fixed batch size; pad all to same length | Wasted compute on padding tokens |
| **Dynamic / sorted batching** | Group by length before batching; less padding | Requires sorting queue; adds latency |
| **Continuous batching** | Insert/evict sequences mid-batch as slots free up | Complex scheduler; vLLM, TGI implement this |

### The padding waste problem

When sequences in a batch have different lengths, shorter ones are padded. The model still attends over pad tokens (masked), wasting FLOPs.

$$\text{Padding waste\%} = 1 - \frac{\sum_i \text{len}_i}{B \times \max_i \text{len}_i}$$

### Continuous Batching (vLLM / PagedAttention)
Traditional approaches pre-allocate the **maximum** possible KV cache per sequence, leading to internal fragmentation. **PagedAttention** (vLLM) stores KV blocks in non-contiguous pages — like virtual memory — enabling near-zero wasted VRAM and >20× throughput on production workloads.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3a ── Static Batching: throughput vs batch size
# ─────────────────────────────────────────────────────────────────────────────
print("📦 Static Batching benchmark on WikiText-2 ...")
batch_sizes   = [1, 2, 4, 8] if DEVICE == "cuda" else [1, 2]
static_tps    = []
static_lat    = []

for bs in batch_sizes:
    res = benchmark_generation(
        model_baseline, tokenizer,
        prompts=EVAL_PROMPTS[:max(bs * 4, 8)],
        max_new_tokens=32,
        batch_size=bs,
        label=f"batch={bs}",
    )
    static_tps.append(res["throughput_tps"])
    static_lat.append(res["latency_ms"])
    print(f"   batch_size={bs:2d}  "
          f"throughput={res['throughput_tps']:.1f} t/s  "
          f"latency={res['latency_ms']:.1f} ms/sample")

RESULTS["Static Batching"] = {
    "latency_ms"    : static_lat[0],
    "throughput_tps": static_tps[-1],   # best throughput (largest batch)
    "peak_mem_gb"   : RESULTS["Baseline (FP16)"]["peak_mem_gb"],
    "perplexity"    : RESULTS["Baseline (FP16)"]["perplexity"],
    "notes"         : f"batch_size={batch_sizes[-1]}",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3b ── Padding waste visualization & Dynamic batching demo
# ─────────────────────────────────────────────────────────────────────────────

# ── Compute padding waste for random vs sorted batching ───────────────────────
sample_prompts = EVAL_PROMPTS[:32]
tokenizer.pad_token = tokenizer.eos_token
enc_all = tokenizer(sample_prompts, return_tensors="pt",
                    padding=True, truncation=True, max_length=256)
lengths = enc_all["attention_mask"].sum(dim=1).tolist()

def batch_padding_waste(lengths_list: List[int], batch_size: int) -> float:
    """Compute % padding waste for a given ordering & batch size."""
    total_tokens, total_slots = 0, 0
    for i in range(0, len(lengths_list), batch_size):
        batch = lengths_list[i : i + batch_size]
        max_l = max(batch)
        total_slots  += max_l * len(batch)
        total_tokens += sum(batch)
    return 1.0 - total_tokens / total_slots

bs_test     = 8
waste_rand  = batch_padding_waste(lengths, bs_test)
waste_sort  = batch_padding_waste(sorted(lengths), bs_test)

print(f"📊 Padding Waste Analysis (batch_size={bs_test}):")
print(f"   Random order   : {waste_rand*100:.1f}% wasted")
print(f"   Sorted by len  : {waste_sort*100:.1f}% wasted")
print(f"   Waste reduction: {(waste_rand - waste_sort)*100:.1f}pp")

# ── Plot: sequence length distribution & padding in a random batch ────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Length histogram
axes[0].hist(lengths, bins=20, color=COLORS[0], edgecolor="white")
axes[0].set_title("Prompt Length Distribution\n(WikiText-2, 32 samples)", fontweight="bold")
axes[0].set_xlabel("Length (tokens)")
axes[0].set_ylabel("Count")

# Static vs sorted waste
axes[1].bar(["Random\nOrder", "Sorted\nby Length"],
            [waste_rand * 100, waste_sort * 100],
            color=[COLORS[6], COLORS[2]], edgecolor="white")
axes[1].bar_label(axes[1].containers[0], fmt="%.1f%%", padding=4)
axes[1].set_title(f"Padding Waste % (batch={bs_test})", fontweight="bold")
axes[1].set_ylabel("Wasted tokens (%)")

# Throughput vs batch size
axes[2].plot(batch_sizes, static_tps, marker="o",
             color=COLORS[3], linewidth=2.5)
axes[2].set_title("Throughput vs. Batch Size\n(Static Batching)", fontweight="bold")
axes[2].set_xlabel("Batch Size")
axes[2].set_ylabel("Tokens / sec")
for x, y in zip(batch_sizes, static_tps):
    axes[2].annotate(f"{y:.0f}", (x, y), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=9)

plt.suptitle("Section 3 — Batching Analysis on WikiText-2",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3c ── Continuous Batching with vLLM (or conceptual demo)
# ─────────────────────────────────────────────────────────────────────────────
try:
    from vllm import LLM, SamplingParams

    print("🚀 Running vLLM continuous batching benchmark ...")
    llm = LLM(model=MODEL_ID, dtype="float16", max_model_len=512)
    params = SamplingParams(max_tokens=64, temperature=0)

    reset_peak_mem()
    sync()
    t0 = time.perf_counter()
    outputs = llm.generate(EVAL_PROMPTS[:20], params)
    sync()
    elapsed = time.perf_counter() - t0

    n_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    vllm_tps = n_new_tokens / elapsed

    print(f"\n📊 vLLM Continuous Batching:")
    print(f"   Throughput         : {vllm_tps:.1f} tokens/sec")
    print(f"   Total elapsed      : {elapsed:.2f} s  (20 prompts, 64 new tokens)")
    hf_tps = RESULTS["Static Batching"]["throughput_tps"]
    print(f"   vs HF Static batch : {vllm_tps/hf_tps:.2f}× speedup")

    RESULTS["vLLM Continuous"] = {
        "latency_ms"    : elapsed * 1000 / 20,
        "throughput_tps": vllm_tps,
        "peak_mem_gb"   : peak_gpu_mem_gb(),
        "perplexity"    : RESULTS["Baseline (FP16)"]["perplexity"],
        "notes"         : "PagedAttention, continuous batching",
    }
    del llm; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

except ImportError:
    print("⚠️  vLLM not installed — showing conceptual comparison only.\n")
    print("Conceptual vLLM vs. HuggingFace Throughput Comparison:")
    print("─" * 50)
    hf_tps = RESULTS.get("Static Batching", {}).get("throughput_tps", 1)
    conceptual = {
        "HuggingFace\n(static batch)": hf_tps,
        "vLLM\n(continuous)": hf_tps * 4.5,   # typical 3-5× reported improvement
        "TGI\n(continuous)": hf_tps * 3.8,
    }
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(list(conceptual.keys()), list(conceptual.values()),
                  color=[COLORS[0], COLORS[2], COLORS[4]], edgecolor="white")
    ax.bar_label(bars, fmt="%.0f t/s", padding=4)
    ax.set_title("Conceptual Throughput: Static vs Continuous Batching\n"
                 "(Install vLLM to see real numbers)", fontweight="bold")
    ax.set_ylabel("Tokens / sec")
    plt.tight_layout(); plt.show()
    
    RESULTS["vLLM Continuous"] = {
        "latency_ms"    : None,
        "throughput_tps": hf_tps * 4.5,
        "peak_mem_gb"   : None,
        "perplexity"    : None,
        "notes"         : "Conceptual estimate (vLLM not installed)",
    }

### 📝 Assignment 3 — Optimal Batch Size Under a Memory Budget

**Task:** Given a fixed GPU memory budget (simulate 8 GB for a single request queue):

1. Sweep `batch_size ∈ [1, 2, 4, 8, 16, 32]` and record:
   - Throughput (tokens/sec)
   - Peak GPU memory (GB)
   - OOM indicator (True/False)

2. Plot **throughput vs. batch size**, marking the largest batch that fits in 8 GB.

3. Compute **GPU utilization** = throughput / theoretical_peak_bandwidth, where:
   $$\text{theoretical peak} = \frac{\text{memory bandwidth (GB/s)}}{\text{model size (GB)}} \times \text{vocab size}$$

4. What is the **saturation point** — the batch size after which throughput gains flatten? Why does it plateau?

> 💡 Hint: Wrap the benchmark call in `try/except torch.cuda.OutOfMemoryError` to handle OOM gracefully. Always call `torch.cuda.empty_cache()` after an OOM.

In [ ]:
# ── Assignment 3 Starter Code ─────────────────────────────────────────────────
a3_batch_sizes = [1, 2, 4, 8, 16, 32] if DEVICE == "cuda" else [1, 2, 4]
a3_tps, a3_mem, a3_oom = [], [], []

for bs in a3_batch_sizes:
    try:
        res = benchmark_generation(
            model_baseline, tokenizer,
            prompts=EVAL_PROMPTS[:max(bs * 3, 12)],
            max_new_tokens=32,
            batch_size=bs,
            label=f"bs={bs}",
        )
        a3_tps.append(res["throughput_tps"])
        a3_mem.append(res["peak_mem_gb"])
        a3_oom.append(False)
        print(f"   bs={bs:3d}  tps={res['throughput_tps']:.1f}  mem={res['peak_mem_gb']:.2f} GB")
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        print(f"   bs={bs:3d}  ❌ OOM: {str(e)[:50]}")
        a3_tps.append(0); a3_mem.append(float("nan")); a3_oom.append(True)
        if DEVICE == "cuda": torch.cuda.empty_cache()

# ── YOUR CODE HERE ─────────────────────────────────────────────────────────────
# Plot: throughput vs batch size and memory vs batch size
# Mark the saturation point and the OOM batch sizes
print("\n✅ Assignment 3 scaffold done — add your plots and analysis above")

---
## Section 4 — Quantization: FP32 → FP16 → INT8 → INT4

### Why Quantize?

LLMs are memory-bandwidth bound. Reducing weight precision directly reduces:
- **Model size on disk**
- **GPU HBM traffic per token** (the bottleneck!)
- **Peak VRAM** needed for serving

| Precision | Bytes/param | Llama-3.2-1B size | Notes |
|-----------|-------------|-------------------|-------|
| **FP32**  | 4 B         | ~4.8 GB           | Training default |
| **FP16**  | 2 B         | ~2.4 GB           | Standard serving |
| **INT8**  | 1 B         | ~1.2 GB           | `bitsandbytes` LLM.int8() |
| **NF4/INT4** | 0.5 B    | ~0.6 GB           | GGUF, GPTQ, AWQ, QLoRA |

### Quantization Methods (Industry Overview)

| Method | Type | Key Idea |
|--------|------|----------|
| **LLM.int8()** | PTQ, weight-only | Mixed-precision: keeps outlier channels in FP16 |
| **GPTQ** | PTQ, weight-only | Second-order gradient optimization of rounding error |
| **AWQ** | PTQ, weight-only | Activations-aware; scales salient channels before rounding |
| **NF4** (QLoRA) | PTQ, NormalFloat | Optimal quantization grid for normally distributed weights |

### Perplexity as quality proxy

We use **WikiText-2 perplexity** to measure quality degradation. A perplexity increase of < 0.5 ppl is generally acceptable for production.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 ── Load models at different precisions & benchmark
# ─────────────────────────────────────────────────────────────────────────────

def load_model_precision(precision: str):
    """Load LLaMA-3.2-1B at specified precision using bitsandbytes."""
    kwargs = dict(device_map="auto" if DEVICE == "cuda" else None)

    if precision == "fp32":
        kwargs["torch_dtype"] = torch.float32
    elif precision == "fp16":
        kwargs["torch_dtype"] = torch.float16
    elif precision == "int8":
        if DEVICE != "cuda":
            print("⚠️  INT8 requires CUDA – skipping")
            return None
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif precision == "int4":
        if DEVICE != "cuda":
            print("⚠️  INT4 requires CUDA – skipping")
            return None
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        raise ValueError(f"Unknown precision: {precision}")

    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    if DEVICE == "cpu" and precision in ("fp32", "fp16"):
        m = m.to(DEVICE)
    m.eval()
    return m


quant_results = {}
precisions    = (["fp16", "int8", "int4"] if DEVICE == "cuda"
                 else ["fp32", "fp16"])

eval_prompts_quant = EVAL_PROMPTS[:10]   # small slice for speed

for prec in precisions:
    print(f"\n⚙️  Loading {prec.upper()} model ...")
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

    m = load_model_precision(prec)
    if m is None:
        continue

    # ── Size ─────────────────────────────────────────────────────────────────
    size_gb = sum(p.numel() * p.element_size() for p in m.parameters()) / 1e9

    # ── Benchmark ────────────────────────────────────────────────────────────
    res = benchmark_generation(m, tokenizer,
                               prompts=eval_prompts_quant,
                               max_new_tokens=32,
                               batch_size=1,
                               label=prec.upper())

    # ── Perplexity ────────────────────────────────────────────────────────────
    ppl = compute_perplexity(m, tokenizer, eval_prompts_quant)

    quant_results[prec.upper()] = {
        "size_gb"       : size_gb,
        "latency_ms"    : res["latency_ms"],
        "throughput_tps": res["throughput_tps"],
        "peak_mem_gb"   : res["peak_mem_gb"],
        "perplexity"    : ppl,
    }

    print(f"   Size       : {size_gb:.2f} GB")
    print(f"   Latency    : {res['latency_ms']:.1f} ms")
    print(f"   Throughput : {res['throughput_tps']:.1f} t/s")
    print(f"   Perplexity : {ppl:.2f}")

    RESULTS[f"Quantization ({prec.upper()})"] = {
        "latency_ms"    : res["latency_ms"],
        "throughput_tps": res["throughput_tps"],
        "peak_mem_gb"   : res["peak_mem_gb"],
        "perplexity"    : ppl,
        "notes"         : f"bitsandbytes {prec}",
    }

    del m; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

print("\n✅ Quantization benchmarks complete")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 ── Quantization results visualization
# ─────────────────────────────────────────────────────────────────────────────
if quant_results:
    labels = list(quant_results.keys())
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    metrics = [
        ("size_gb",        "Model Size (GB)",       "GB",    "lower is better"),
        ("latency_ms",     "Latency / Sample (ms)", "ms",    "lower is better"),
        ("throughput_tps", "Throughput (t/s)",      "t/s",   "higher is better"),
        ("perplexity",     "WikiText-2 Perplexity", "ppl",   "lower is better"),
    ]

    palette = [COLORS[i % len(COLORS)] for i in range(len(labels))]

    for ax, (key, title, unit, hint) in zip(axes.flat, metrics):
        vals  = [quant_results[l][key] for l in labels]
        bars  = ax.bar(labels, vals, color=palette, edgecolor="white")
        ax.bar_label(bars, fmt=f"%.2f", padding=4, fontsize=10)
        ax.set_title(f"{title}\n({hint})", fontweight="bold")
        ax.set_ylabel(unit)

    plt.suptitle("Quantization Comparison on WikiText-2 (LLaMA-3.2-1B)",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    df_quant = pd.DataFrame(quant_results).T
    df_quant.index.name = "Precision"
    df_quant["compression_vs_fp16"] = (
        quant_results.get("FP16", quant_results[list(quant_results.keys())[0]])["size_gb"]
        / df_quant["size_gb"]
    ).round(2)
    print("\n📋 Quantization Summary Table:")
    print(df_quant[["size_gb","latency_ms","throughput_tps","perplexity","compression_vs_fp16"]]
          .to_string())
else:
    print("⚠️  No quantization results (CUDA required for INT8/INT4).")

### 📝 Assignment 4 — Quality-Efficiency Tradeoff Analysis

**Task:** Identify the ideal quantization precision for a production deployment.

1. Using the results above, plot the **perplexity vs. compression ratio** curve (x = compression ratio vs FP16, y = WikiText-2 perplexity).

2. Mark a "knee point" — the precision at which perplexity starts to increase significantly.

3. Compute the **efficiency score** for each precision:
$$\text{Score} = \frac{\text{throughput (t/s)}}{\text{perplexity} \times \text{model size (GB)}}$$
Higher = more efficient use of GPU memory for quality.

4. For each of the following deployment scenarios, which precision would you choose and why?
   - **A)** Real-time chatbot (latency < 200 ms, quality critical)
   - **B)** Batch summarization pipeline (throughput priority, slight quality loss OK)
   - **C)** On-device mobile inference (< 2 GB RAM, best-effort quality)

5. **Advanced**: Look up **GPTQ** and **AWQ**. How do they achieve better perplexity than naive round-to-nearest INT4? Write a 3-sentence explanation.

In [ ]:
# ── Assignment 4 Starter Code ─────────────────────────────────────────────────
if quant_results:
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────
    # 1. Compute compression ratio for each precision relative to FP16
    fp16_size = quant_results.get("FP16", list(quant_results.values())[0])["size_gb"]

    for prec, vals in quant_results.items():
        compression   = fp16_size / vals["size_gb"]
        eff_score     = vals["throughput_tps"] / (vals["perplexity"] * vals["size_gb"])
        print(f"  {prec:8s}  compression={compression:.2f}x  "
              f"eff_score={eff_score:.3f}  ppl={vals['perplexity']:.2f}")

    # 2. YOUR CODE: plot perplexity vs compression ratio
    # 3. YOUR CODE: mark deployment scenario choices (A, B, C) on the chart

    print("\n✅ Assignment 4 scaffold ready")

---
## Section 5 — Flash Attention

### Standard Attention Memory Problem

Standard scaled dot-product attention requires materializing the full $N \times N$ attention matrix:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

For a sequence of length $N$:
- **Memory**: $O(N^2)$ — a 4K-token sequence with 32 heads = 4 GB of attention maps!
- **HBM reads/writes**: Huge — softmax requires reading the full attention matrix multiple times

### FlashAttention (Dao et al., 2022)

**Key insight**: Instead of materializing the full matrix, use **online softmax** + **tiling** to compute attention in blocks that fit in SRAM. Recompute blocks during the backward pass instead of storing them.

| Property | Standard Attention | FlashAttention v2 |
|---------|-------------------|-------------------|
| Memory | $O(N^2)$ | $O(N)$ |
| HBM reads | $O(N^2)$ | $O(N^2 / M)$ where $M$ = SRAM size |
| Speed (A100) | 1× | ~2–4× faster |
| Exact? | Yes | Yes ✅ |

### PyTorch 2.0 SDPA (fallback)
Without `flash-attn`, PyTorch 2.0+ provides `torch.nn.functional.scaled_dot_product_attention` with optimized CUDA kernels (memory-efficient attention). We use this as our comparison point.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5a ── Flash Attention benchmark: memory & speed across seq lengths
# ─────────────────────────────────────────────────────────────────────────────
flash_seq_lengths = [128, 256, 512, 1024, 2048]
if DEVICE != "cuda":
    flash_seq_lengths = [128, 256, 512]    # CPU: keep it short

results_sdpa  = {"seq_len": flash_seq_lengths, "time_ms": [], "mem_gb": []}
results_flash = {"seq_len": flash_seq_lengths, "time_ms": [], "mem_gb": []}

# ── Standard SDPA (PyTorch 2.0) ───────────────────────────────────────────────
print("📐 Benchmarking Standard SDPA (PyTorch 2.0) ...")
model_sdpa = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    attn_implementation="sdpa",
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE == "cpu":
    model_sdpa = model_sdpa.to(DEVICE)
model_sdpa.eval()

for slen in flash_seq_lengths:
    ids  = torch.tensor([tokens_src[:slen]], device=DEVICE)
    attn = torch.ones_like(ids)
    reset_peak_mem()

    # warm-up
    with torch.no_grad():
        _ = model_sdpa(input_ids=ids, attention_mask=attn)

    sync()
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model_sdpa(input_ids=ids, attention_mask=attn)
        sync()
        times.append((time.perf_counter() - t0) * 1000)

    results_sdpa["time_ms"].append(float(np.mean(times)))
    results_sdpa["mem_gb"].append(peak_gpu_mem_gb())
    print(f"   SDPA  seq={slen:5d}  {np.mean(times):.1f} ms  "
          f"mem={peak_gpu_mem_gb():.2f} GB")

del model_sdpa; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

# ── Flash Attention 2 ─────────────────────────────────────────────────────────
print("\n⚡ Benchmarking Flash Attention 2 ...")
try:
    model_flash = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,   # FlashAttn requires FP16 or BF16
        attn_implementation="flash_attention_2",
        device_map="auto",
    )
    model_flash.eval()
    flash_available = True
except Exception as e:
    print(f"⚠️  Flash Attention 2 not available ({e}). Using PyTorch SDPA as proxy.")
    model_flash = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE,
        attn_implementation="sdpa",
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE == "cpu": model_flash = model_flash.to(DEVICE)
    model_flash.eval()
    flash_available = False

for slen in flash_seq_lengths:
    ids  = torch.tensor([tokens_src[:slen]], device=DEVICE)
    attn = torch.ones_like(ids)
    reset_peak_mem()

    with torch.no_grad():
        _ = model_flash(input_ids=ids, attention_mask=attn)  # warm-up

    sync()
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model_flash(input_ids=ids, attention_mask=attn)
        sync()
        times.append((time.perf_counter() - t0) * 1000)

    results_flash["time_ms"].append(float(np.mean(times)))
    results_flash["mem_gb"].append(peak_gpu_mem_gb())
    label = "Flash2" if flash_available else "SDPA(2)"
    print(f"   {label} seq={slen:5d}  {np.mean(times):.1f} ms  "
          f"mem={peak_gpu_mem_gb():.2f} GB")

del model_flash; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

RESULTS["Flash Attention"] = {
    "latency_ms"    : results_flash["time_ms"][-1],
    "throughput_tps": 1000 / max(results_flash["time_ms"][-1], 1e-6),
    "peak_mem_gb"   : results_flash["mem_gb"][-1],
    "perplexity"    : RESULTS["Baseline (FP16)"]["perplexity"],
    "notes"         : "FlashAttn2 or SDPA fallback",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5b ── Flash Attention visualization
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
flash_label = "Flash Attention 2" if flash_available else "PyTorch SDPA (v2)"

# ── Latency comparison ────────────────────────────────────────────────────────
axes[0].plot(results_sdpa["seq_len"],  results_sdpa["time_ms"],
             marker="o", label="Standard SDPA (PyTorch 2.0)", color=COLORS[0], linewidth=2)
axes[0].plot(results_flash["seq_len"], results_flash["time_ms"],
             marker="s", label=flash_label, color=COLORS[2], linewidth=2)
axes[0].set_title("Forward Pass Latency vs Sequence Length", fontweight="bold")
axes[0].set_xlabel("Sequence Length (tokens)")
axes[0].set_ylabel("Time (ms)")
axes[0].legend()

# ── Memory comparison ─────────────────────────────────────────────────────────
axes[1].plot(results_sdpa["seq_len"],  results_sdpa["mem_gb"],
             marker="o", label="Standard SDPA", color=COLORS[0], linewidth=2)
axes[1].plot(results_flash["seq_len"], results_flash["mem_gb"],
             marker="s", label=flash_label, color=COLORS[2], linewidth=2)
axes[1].set_title("Peak GPU Memory vs Sequence Length", fontweight="bold")
axes[1].set_xlabel("Sequence Length (tokens)")
axes[1].set_ylabel("GPU Memory (GB)")
axes[1].legend()

# Speedup annotation
speedups = [s/f if f > 0 else 1
            for s, f in zip(results_sdpa["time_ms"], results_flash["time_ms"])]
mid = len(flash_seq_lengths) // 2
axes[0].annotate(
    f"~{speedups[mid]:.1f}× faster",
    xy=(flash_seq_lengths[mid], results_flash["time_ms"][mid]),
    xytext=(flash_seq_lengths[mid] // 2, results_flash["time_ms"][mid] * 2.2),
    arrowprops=dict(arrowstyle="->", color="crimson"),
    color="crimson", fontsize=11
)

plt.suptitle("Section 5 — Flash Attention on WikiText-2",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Memory savings table
print("\n📊 Memory Savings from Flash Attention:")
print(f"  {'Seq':>6s}  {'SDPA (GB)':>10s}  {'Flash (GB)':>11s}  {'Saving':>8s}")
for i, seqlen in enumerate(flash_seq_lengths):
    sdpa_m  = results_sdpa["mem_gb"][i]
    flash_m = results_flash["mem_gb"][i]
    saving  = (sdpa_m - flash_m) / max(sdpa_m, 1e-6) * 100
    print(f"  {seqlen:>6d}  {sdpa_m:>10.3f}  {flash_m:>11.3f}  {saving:>7.1f}%")

### 📝 Assignment 5 — When Does Flash Attention Become Essential?

**Task:**

1. Extend the benchmark to sequence lengths `[128, 256, 512, 1024, 2048, 4096]`.

2. For standard attention, derive the **theoretical** memory requirement:
   $$M_{\text{attn}} = n_{\text{layers}} \times n_{\text{heads}} \times N^2 \times 2 \text{ bytes (FP16)}$$

   Plot this theoretical curve alongside your measured values. Do they match?

3. At what sequence length does standard attention **memory exceed Flash Attention memory by more than 2×**?

4. If your GPU has **24 GB VRAM** and the model weights occupy 2.4 GB, what is the maximum sequence length you can process with:
   - Standard attention?
   - Flash Attention?

   Show your calculation.

5. **Bonus**: Read about **FlashAttention-3** (2024). What new GPU features does it exploit vs. FlashAttention-2? Write a 2-sentence summary.

---
## Section 6 — Speculative Decoding

### The Bottleneck: Sequential Decode Steps

In standard decoding, each token requires a full forward pass through the **target model** (large). These passes are sequential and memory-bandwidth bound.

### The Speculative Decoding Idea (Leviathan et al., 2022)

Use a **small, fast draft model** $M_q$ to propose $\gamma$ speculative tokens, then verify *all* of them in **one single forward pass** of the large target model $M_p$.

**Algorithm:**
1. Draft model $M_q$ auto-regressively generates $\gamma$ candidate tokens
2. Target model $M_p$ processes the prompt + all $\gamma$ candidates **in parallel** (like a prefill)
3. Each candidate is **accepted** if the target model agrees (based on acceptance probability $\alpha$)
4. The first rejected token is resampled from the target; decoding resumes

**Expected speedup:**

$$\text{speedup} = \frac{1 - \alpha^\gamma}{(1 - \alpha)(1 + \frac{c}{\gamma})}$$

where $\alpha$ = acceptance rate, $\gamma$ = draft tokens per step, $c$ = cost ratio (draft / target forward pass). Typical: **2–3× decode throughput** with $\alpha \approx 0.8$, $\gamma = 4$.

### Why it works for WikiText-2

WikiText-2 contains formal, predictable English text. The draft model often predicts the next few tokens correctly, achieving high acceptance rates.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6a ── Load draft (1B) and target (3B) models
# ─────────────────────────────────────────────────────────────────────────────
spec_available = DEVICE == "cuda"

if spec_available:
    print(f"📥 Loading draft model : {MODEL_ID}")
    model_draft = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto")
    model_draft.eval()

    print(f"📥 Loading target model: {MODEL_ID_LARGE}")
    try:
        model_target = AutoModelForCausalLM.from_pretrained(
            MODEL_ID_LARGE, torch_dtype=torch.float16, device_map="auto")
        model_target.eval()
        print("✅ Both models loaded")
    except Exception as e:
        print(f"⚠️  Could not load large model ({e})")
        print("   Using 1B as both draft and target for demo purposes.")
        model_target = model_draft
else:
    print("⚠️  Speculative decoding requires CUDA. Showing theoretical analysis only.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6b ── Speculative decoding: HuggingFace assisted generation
# ─────────────────────────────────────────────────────────────────────────────
spec_prompts    = EVAL_PROMPTS[:10]
max_new_tokens  = 64

if spec_available:
    # ── Standard (target only) ───────────────────────────────────────────────
    print("⏱️  Standard decoding (target model only) ...")
    reset_peak_mem()
    sync(); t0 = time.perf_counter()
    std_outputs, std_tokens = [], 0
    for prompt in tqdm(spec_prompts, desc="  [standard]"):
        enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model_target.generate(
                **enc, max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_t = out.shape[-1] - enc["input_ids"].shape[-1]
        std_tokens += new_t
        std_outputs.append(tokenizer.decode(out[0, enc["input_ids"].shape[-1]:],
                                             skip_special_tokens=True))
    sync(); std_elapsed = time.perf_counter() - t0
    std_tps = std_tokens / std_elapsed
    print(f"   Standard  : {std_tps:.1f} t/s  ({std_elapsed:.2f}s total)")

    # ── Speculative (draft + target) ─────────────────────────────────────────
    print("🚀 Speculative decoding (draft → target verify) ...")
    reset_peak_mem()
    sync(); t1 = time.perf_counter()
    spec_tokens = 0
    for prompt in tqdm(spec_prompts, desc="  [speculative]"):
        enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model_target.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                assistant_model=model_draft,   # ← speculative decoding flag
                num_assistant_tokens=4,         # γ = 4 draft tokens per step
            )
        spec_tokens += out.shape[-1] - enc["input_ids"].shape[-1]
    sync(); spec_elapsed = time.perf_counter() - t1
    spec_tps = spec_tokens / spec_elapsed
    speedup  = spec_tps / std_tps
    print(f"   Speculative: {spec_tps:.1f} t/s  ({spec_elapsed:.2f}s total)")
    print(f"\n   🏎️  Speedup: {speedup:.2f}×")

    RESULTS["Speculative Decoding"] = {
        "latency_ms"    : spec_elapsed * 1000 / len(spec_prompts),
        "throughput_tps": spec_tps,
        "peak_mem_gb"   : peak_gpu_mem_gb(),
        "perplexity"    : RESULTS["Baseline (FP16)"]["perplexity"],  # same quality
        "notes"         : f"draft=1B, target=3B, γ=4",
    }
else:
    # ── Theoretical speedup curves ───────────────────────────────────────────
    print("📊 Theoretical Speculative Decoding Analysis\n")
    alpha_values = [0.6, 0.7, 0.8, 0.9]
    gamma_values = list(range(1, 9))
    cost_ratio   = 0.15   # draft is ~15% cost of target

    fig, ax = plt.subplots(figsize=(9, 5))
    for alpha in alpha_values:
        speedups_theory = []
        for gamma in gamma_values:
            num = 1 - alpha**gamma
            den = (1 - alpha) * (1 + cost_ratio / gamma)
            su  = num / den if den > 0 else 1
            speedups_theory.append(su / (1 + cost_ratio))   # normalize vs no-draft
        ax.plot(gamma_values, speedups_theory, marker="o",
                label=f"α={alpha}", linewidth=2)

    ax.axhline(1.0, color="grey", linestyle="--", alpha=0.7, label="No Speculation")
    ax.set_title("Theoretical Speedup of Speculative Decoding\n"
                 "(CUDA required for empirical run)", fontweight="bold")
    ax.set_xlabel("γ (draft tokens per step)")
    ax.set_ylabel("Speedup vs. standard decoding")
    ax.legend(title="Acceptance rate α")
    plt.tight_layout()
    plt.show()
    print("⚠️  Run on GPU to get empirical results.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6c ── γ sweep: draft tokens per step vs throughput
# ─────────────────────────────────────────────────────────────────────────────
if spec_available:
    gamma_values = [1, 2, 4, 8]
    gamma_tps    = []

    for gamma in gamma_values:
        sync(); t0 = time.perf_counter()
        tot_new = 0
        for prompt in tqdm(spec_prompts[:5], desc=f"  [γ={gamma}]", leave=False):
            enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                out = model_target.generate(
                    **enc, max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    assistant_model=model_draft,
                    num_assistant_tokens=gamma,
                )
            tot_new += out.shape[-1] - enc["input_ids"].shape[-1]
        sync()
        elapsed = time.perf_counter() - t0
        tps = tot_new / elapsed
        gamma_tps.append(tps)
        print(f"   γ={gamma}  throughput={tps:.1f} t/s")

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(gamma_values, gamma_tps, marker="D",
            color=COLORS[4], linewidth=2.5, markersize=9)
    ax.axhline(std_tps, color="grey", linestyle="--", label=f"Standard ({std_tps:.0f} t/s)")
    ax.bar_label(ax.bar(gamma_values, gamma_tps, alpha=0),
                 labels=[f"{v:.0f}" for v in gamma_tps], padding=6)
    ax.set_title("Speculative Decoding: Throughput vs γ\n(WikiText-2)", fontweight="bold")
    ax.set_xlabel("γ (draft tokens per step)")
    ax.set_ylabel("Tokens / sec")
    ax.legend()
    plt.tight_layout()
    plt.show()

    opt_gamma = gamma_values[int(np.argmax(gamma_tps))]
    print(f"\n🏆 Optimal γ for WikiText-2: {opt_gamma} "
          f"({max(gamma_tps):.1f} t/s)")
else:
    print("⚠️  γ sweep requires CUDA — skipped in CPU mode.")

### 📝 Assignment 6 — Acceptance Rate and Optimal γ

**Task:**

1. Implement a **manual speculative decoding loop** that tracks the **acceptance rate** $\alpha$ per prompt:

```python
def speculative_decode_with_stats(draft_model, target_model, tokenizer,
                                   prompt, gamma=4, max_new_tokens=64):
    # Returns: (generated_text, tokens_per_sec, acceptance_rate)
    ...
```

2. Run it on 10 WikiText-2 prompts with `γ ∈ [1, 2, 4, 8]`. Record:
   - Mean acceptance rate $\alpha$
   - Effective throughput (tokens/sec)

3. Plot **α vs. γ** and **throughput vs. γ** side by side.

4. Using the theoretical speedup formula:
   $$\text{speedup} = \frac{1 - \alpha^{\gamma+1}}{(1-\alpha)(1 + c \cdot \gamma)}$$
   Back-calculate the empirical cost ratio $c = T_{\text{draft}} / T_{\text{target}}$ from your measurements.

5. **Discussion**: Speculative decoding works differently for code vs. natural language vs. factual QA. Why? Which WikiText-2 text type (narrative, lists, technical) would you expect to yield the highest $\alpha$?

> 💡 **Acceptance rule**: At position $i$, accept draft token $x$ if:
> $$u \sim \text{Uniform}[0,1], \quad u \leq \frac{p_{\text{target}}(x)}{p_{\text{draft}}(x)}$$

In [ ]:
# ── Assignment 6 Starter Code ─────────────────────────────────────────────────
def speculative_decode_with_stats(
    draft_model, target_model, tokenizer,
    prompt: str, gamma: int = 4, max_new_tokens: int = 64
):
    """
    Manual speculative decoding with acceptance rate tracking.
    Returns (generated_text, tokens_per_sec, acceptance_rate)
    """
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids = enc["input_ids"]
    accepted = 0
    total_draft = 0
    generated = []

    sync(); t_start = time.perf_counter()

    while len(generated) < max_new_tokens:
        # ── Step 1: Draft γ tokens ─────────────────────────────────────────
        draft_tokens = []
        draft_logprobs = []
        cur_ids = input_ids

        for _ in range(gamma):
            with torch.no_grad():
                draft_out = draft_model(input_ids=cur_ids)
            logits = draft_out.logits[:, -1, :]
            probs  = torch.softmax(logits, dim=-1)
            next_tok = probs.argmax(dim=-1, keepdim=True)
            draft_tokens.append(next_tok)
            draft_logprobs.append(probs[0, next_tok.item()].item())
            cur_ids = torch.cat([cur_ids, next_tok], dim=-1)
            if next_tok.item() == tokenizer.eos_token_id:
                break

        total_draft += len(draft_tokens)

        # ── Step 2: Target verifies all at once ───────────────────────────
        full_ids = torch.cat([input_ids] + draft_tokens, dim=-1)
        with torch.no_grad():
            target_out = target_model(input_ids=full_ids)

        # ── YOUR CODE HERE ────────────────────────────────────────────────
        # Step 3: Accept/reject each draft token using the speculative rule:
        #   u ~ Uniform[0,1]; accept if u <= p_target / p_draft
        # Append accepted tokens to `generated`; resample the first rejected.
        # Break if EOS is accepted or max_new_tokens reached.

        # For now, accept all (incorrect — your task is to implement rejection)
        for tok in draft_tokens:
            generated.append(tok.item())
            accepted += 1

        # Advance input_ids
        input_ids = full_ids
        if len(generated) >= max_new_tokens:
            break

    sync()
    elapsed = time.perf_counter() - t_start
    accept_rate = accepted / max(total_draft, 1)
    tps = len(generated) / max(elapsed, 1e-6)
    text = tokenizer.decode(generated[:max_new_tokens], skip_special_tokens=True)
    return text, tps, accept_rate


# ── Quick sanity test ─────────────────────────────────────────────────────────
if spec_available:
    text, tps, acc = speculative_decode_with_stats(
        model_draft, model_target, tokenizer,
        EVAL_PROMPTS[0], gamma=4, max_new_tokens=16)
    print(f"Generated  : {text[:80]}")
    print(f"TPS        : {tps:.1f}")
    print(f"Accept rate: {acc:.2f}  (should be < 1.0 after implementing rejection)")
    print("\n✅ Implement the rejection sampling step above!")
else:
    print("⚠️  Requires CUDA — implement and run on a GPU instance.")

---
## Section 7 — Summary Dashboard

This section aggregates all benchmark results and produces:
1. A **comparison DataFrame** across all techniques
2. A **radar chart** showing the multi-dimensional tradeoff
3. A **decision guide** for choosing the right optimization

### Key Takeaways

| Optimization | When to use |
|---|---|
| **KV Cache** | Always — no quality cost, 2–5× decode speedup |
| **INT8 Quantization** | Production serving when VRAM is tight; < 0.5 ppl degradation |
| **INT4 (NF4)** | Edge/mobile; 4× compression at minor quality cost |
| **Flash Attention** | Any long-context use case (> 512 tokens) |
| **Batching** | Offline workloads; significant throughput gain at batch ≥ 8 |
| **Speculative Decoding** | Interactive chat with predictable text; 2–3× latency reduction |
| **Continuous Batching (vLLM)** | High-traffic production APIs; eliminates HOL blocking |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7a ── Build comparison DataFrame from all RESULTS
# ─────────────────────────────────────────────────────────────────────────────
print("📋 Full Results Table:")
print("─" * 85)

rows = []
for name, vals in RESULTS.items():
    rows.append({
        "Method"          : name,
        "Latency (ms)"    : round(vals.get("latency_ms") or 0, 1),
        "Throughput (t/s)": round(vals.get("throughput_tps") or 0, 1),
        "Peak Mem (GB)"   : round(vals.get("peak_mem_gb") or 0, 2),
        "Perplexity"      : round(vals.get("perplexity") or 0, 2) if vals.get("perplexity") else "N/A",
        "Notes"           : vals.get("notes", ""),
    })

df = pd.DataFrame(rows).set_index("Method")
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.width", 120)
print(df.to_string())

# Highlight best values
print("\n🏆 Best Latency    :", df["Latency (ms)"][df["Latency (ms)"] > 0].idxmin())
print("🏆 Best Throughput :", df["Throughput (t/s)"].idxmax())
print("🏆 Best Memory     :", df["Peak Mem (GB)"][df["Peak Mem (GB)"] > 0].idxmin())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7b ── Side-by-side bar comparison: latency, throughput, memory
# ─────────────────────────────────────────────────────────────────────────────
# Filter to methods with numeric results
df_plot = df[df["Throughput (t/s)"] > 0].copy()
methods = df_plot.index.tolist()
palette = [COLORS[i % len(COLORS)] for i in range(len(methods))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, ylabel, hint) in zip(axes, [
    ("Latency (ms)",     "ms / sample",   "lower is better"),
    ("Throughput (t/s)", "tokens / sec",  "higher is better"),
    ("Peak Mem (GB)",    "GB",            "lower is better"),
]):
    vals = df_plot[col].tolist()
    bars = ax.bar(range(len(methods)), vals, color=palette, edgecolor="white")
    ax.bar_label(bars, fmt="%.1f", padding=4, fontsize=9)
    ax.set_title(f"{col}\n({hint})", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in methods],
                       fontsize=8, rotation=0)

plt.suptitle("LLM Inference Optimization — Full Comparison (WikiText-2 / LLaMA-3.2-1B)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7c ── Radar chart: normalized multi-dimensional comparison
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from matplotlib.patches import FancyArrowPatch

dimensions = ["Throughput", "Low Latency", "Low Memory", "Quality (PPL)"]

# Helper: normalize 0–1; 1 = best
def normalize_col(vals, higher_is_better=True):
    arr = np.array([v if isinstance(v, (int, float)) and v > 0 else np.nan
                    for v in vals], dtype=float)
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if mx == mn:
        return np.ones_like(arr)
    normed = (arr - mn) / (mx - mn)
    return normed if higher_is_better else (1 - normed)

methods_radar = list(RESULTS.keys())[:6]   # top-6 for clarity
tps_vals  = [RESULTS[m].get("throughput_tps") or 0 for m in methods_radar]
lat_vals  = [RESULTS[m].get("latency_ms")     or 999 for m in methods_radar]
mem_vals  = [RESULTS[m].get("peak_mem_gb")    or 99  for m in methods_radar]
ppl_vals  = [RESULTS[m].get("perplexity")     or 999 for m in methods_radar]

scores = np.stack([
    normalize_col(tps_vals, higher_is_better=True),   # throughput: higher better
    normalize_col(lat_vals, higher_is_better=False),  # latency: lower better
    normalize_col(mem_vals, higher_is_better=False),  # memory: lower better
    normalize_col(ppl_vals, higher_is_better=False),  # perplexity: lower better
], axis=1)

# ── Radar setup ───────────────────────────────────────────────────────────────
N = len(dimensions)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"polar": True})

colors_radar = [COLORS[i % len(COLORS)] for i in range(len(methods_radar))]
for i, (m, sc) in enumerate(zip(methods_radar, scores)):
    vals_r = sc.tolist() + sc[:1].tolist()
    ax.plot(angles, vals_r, linewidth=2, color=colors_radar[i], label=m)
    ax.fill(angles, vals_r, alpha=0.08, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dimensions, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=8)
ax.set_title("Inference Optimization Techniques\nNormalized Performance Radar",
             fontsize=14, fontweight="bold", y=1.1)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7d ── Decision flowchart (text-based) + Stack recommendation
# ─────────────────────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║          INFERENCE OPTIMIZATION DECISION GUIDE (LLM / Transformers)        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  START HERE                                                                  ║
║  │                                                                           ║
║  ├─► Always enable KV Cache (use_cache=True) ——————————————► FREE speedup   ║
║  │                                                                           ║
║  ├─► Sequence > 512 tokens?                                                  ║
║  │     YES ──► Use Flash Attention 2 (attn_implementation="flash_attention_2")║
║  │                                                                           ║
║  ├─► Memory-constrained? (single GPU, < 24 GB)                              ║
║  │     YES ──► INT8 (negligible quality loss)                                ║
║  │           ──► INT4/NF4 if < 8 GB available (minor quality loss)           ║
║  │                                                                           ║
║  ├─► Multiple requests per second? (API serving)                             ║
║  │     YES ──► Continuous batching (vLLM or TGI)                             ║
║  │           ──► Dynamic batching with sorted lengths                         ║
║  │                                                                           ║
║  ├─► Interactive / real-time chat latency?                                   ║
║  │     YES ──► Speculative decoding (γ=4, small draft model)                 ║
║  │           ──► Combine with INT8/INT4 for max efficiency                   ║
║  │                                                                           ║
║  └─► Production stack recommendation:                                        ║
║        KV Cache + Flash Attention 2 + INT4 (NF4) + vLLM                     ║
║        → Typical result: 5–10× baseline throughput, 4× less VRAM            ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# ── Final perplexity guard (sanity check) ─────────────────────────────────────
baseline_ppl = RESULTS.get("Baseline (FP16)", {}).get("perplexity")
if baseline_ppl and baseline_ppl > 0:
    assert baseline_ppl < 200, \
        f"Baseline perplexity {baseline_ppl:.2f} looks too high — check model loading"
    print(f"✅ Sanity check: Baseline perplexity = {baseline_ppl:.2f} (plausible)")

print("\n🎓 Lab Complete! You have implemented and benchmarked:")
for i, key in enumerate(RESULTS.keys(), 1):
    print(f"   {i}. {key}")

---
## 🏆 Bonus Challenge — Compose an Optimal Stack

**Ultimate Task:** Design an end-to-end production inference pipeline that stacks multiple techniques.

### Requirements
- **Model**: LLaMA-3.2-1B (or larger if you have GPU budget)
- **Dataset**: WikiText-2 test set, 100 prompts, 128 new tokens each
- **Target**: Achieve **perplexity ≤ baseline + 1.0** while maximizing throughput

### Your Stack Must Include At Least:
1. ✅ KV Cache enabled
2. ✅ One quantization level (INT8 or INT4)
3. ✅ Flash Attention 2 (or SDPA)
4. ✅ Batching (batch_size ≥ 4)
5. ✅ (Optional) Speculative decoding

### Deliverables

1. A `build_optimized_model()` function that loads and configures your chosen stack
2. A `run_optimized_benchmark(model, tokenizer, prompts)` function
3. A comparison table: Baseline vs. Your Stack for all metrics
4. A 1-paragraph write-up justifying each choice

### Evaluation Scoring

| Metric | Weight | Target |
|--------|--------|--------|
| WikiText-2 Perplexity | 30% | ≤ baseline + 1.0 |
| Throughput (t/s) | 40% | ≥ 3× baseline |
| Peak GPU Memory | 20% | ≤ 50% of baseline |
| Code quality | 10% | Commented, reproducible |

> 🏆 **Top score**: Achieve all four targets simultaneously on a single A10G (24 GB) GPU.

In [ ]:
# ── Bonus Challenge Starter Code ─────────────────────────────────────────────

def build_optimized_model():
    """
    Load LLaMA-3.2-1B with your chosen optimization stack.
    Modify kwargs to implement your stack.
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # Example skeleton — add/modify as needed:

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,       # INT4 quantization
        attn_implementation="flash_attention_2" if DEVICE == "cuda" else "sdpa",
        torch_dtype=torch.float16,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    model.eval()
    return model


def run_optimized_benchmark(model, tokenizer, prompts, batch_size=4):
    """
    Benchmark your optimized model.
    Returns a dict with latency_ms, throughput_tps, peak_mem_gb, perplexity.
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    res = benchmark_generation(
        model, tokenizer,
        prompts=prompts,
        max_new_tokens=128,
        batch_size=batch_size,
        label="Optimized Stack",
        use_cache=True,
    )
    ppl = compute_perplexity(model, tokenizer, prompts[:10])
    res["perplexity"] = ppl
    return res


# ── Run and compare ───────────────────────────────────────────────────────────
if DEVICE == "cuda":
    print("🚀 Building optimized stack ...")
    gc.collect(); torch.cuda.empty_cache()
    opt_model = build_optimized_model()

    print("⏱️  Running optimized benchmark ...")
    opt_res = run_optimized_benchmark(opt_model, tokenizer, EVAL_PROMPTS[:20])

    # ── Comparison table ──────────────────────────────────────────────────────
    baseline_r = RESULTS["Baseline (FP16)"]
    print("\n📊 Baseline vs. Optimized Stack:")
    print(f"{'Metric':<25s}  {'Baseline':>12s}  {'Optimized':>12s}  {'Delta':>10s}")
    print("-" * 63)
    for metric, fmt, higher_better in [
        ("latency_ms",     "{:.1f} ms",  False),
        ("throughput_tps", "{:.1f} t/s", True),
        ("peak_mem_gb",    "{:.2f} GB",  False),
        ("perplexity",     "{:.2f}",     False),
    ]:
        b_val = baseline_r.get(metric) or 0
        o_val = opt_res.get(metric)    or 0
        if b_val and o_val:
            delta = (o_val / b_val - 1) * 100
            arrow = "▲" if (higher_better and delta > 0) or (not higher_better and delta < 0) else "▼"
            print(f"  {metric.replace('_',' ').title():<23s}  {fmt.format(b_val):>12s}  "
                  f"{fmt.format(o_val):>12s}  {arrow} {abs(delta):.1f}%")

    del opt_model; gc.collect(); torch.cuda.empty_cache()
else:
    print("⚠️  Bonus challenge requires CUDA. Implement build_optimized_model() and run on GPU.")

print("\n🎉 Congratulations on completing the LLM Inference Optimization Lab!")